# 🎯 MASTER Workflow - Complete Interactive Clustering Pipeline

**Purpose**: End-to-end interactive clustering analysis with full control over every step

**What you can do**:
- ✅ Import and inspect raw data
- ✅ Configure preprocessing parameters
- ✅ Detect and handle outliers with visualization
- ✅ Toggle PCA on/off and see the impact
- ✅ Select feature sets (static/dynamic/combined)
- ✅ Run K-Means with adjustable k
- ✅ Run Hierarchical clustering with different linkage methods
- ✅ Run DBSCAN with interactive parameter tuning
- ✅ Compare all algorithms side-by-side
- ✅ Backtrack and re-run sections with different settings

**Navigation**: Each section is independent - jump to any step!

---

## 📑 Table of Contents

1. [Setup & Environment](#section-1)
2. [Data Import & Inspection](#section-2)
3. [Data Preprocessing & Formatting](#section-3)
4. [Outlier Detection & Handling](#section-4)
5. [Feature Selection](#section-5)
6. [PCA Configuration (Toggle)](#section-6)
7. [Data Scaling & Preparation](#section-7)
8. [K-Means Clustering](#section-8)
9. [Hierarchical Clustering](#section-9)
10. [DBSCAN Clustering](#section-10)
11. [Algorithm Comparison](#section-11)
12. [Results Export](#section-12)

---

## 1️⃣ Setup & Environment

Load all required libraries and set up the environment.

In [ ]:
# Core imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up project root
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Clustering algorithms
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# Metrics
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)

# Hierarchical clustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats.mstats import winsorize

# Project modules
from src._01_setup import config_loader
from src._02_preprocessing import data_cleaner
from notebook_utils import (
    setup_notebook, NotebookState,
    display_dataframe_summary,
    plot_cluster_distribution,
    plot_cluster_profiles,
    plot_feature_distributions,
    save_results
)

# Jupyter widgets for interactivity
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("⚠️  ipywidgets not available. Install with: pip install ipywidgets")

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"✅ All imports successful!")
print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"🎨 Widgets Available: {WIDGETS_AVAILABLE}")
print(f"🐍 Python: {sys.version.split()[0]}")

## 2️⃣ Data Import & Inspection

Load configuration and raw data files.

In [ ]:
# Setup notebook and load configuration
cfg, state = setup_notebook(
    title="MASTER Workflow - Complete Pipeline",
    market="germany"
)

# Extract key configuration
market = config_loader.get_value(cfg, 'data', 'market', default='germany')
input_dir = config_loader.get_value(cfg, 'data', 'input_dir', default='data/raw')

print(f"\n📋 Configuration Loaded:")
print(f"  Market: {market}")
print(f"  Input Directory: {input_dir}")
print(f"\n✅ Configuration ready!")

In [ ]:
# Locate data files
data_dir = PROJECT_ROOT / input_dir / market

print(f"📁 Data Directory: {data_dir}")
print(f"\nAvailable files:")

if data_dir.exists():
    files = sorted(data_dir.glob('*.csv'))
    for i, f in enumerate(files, 1):
        size_mb = f.stat().st_size / 1024**2
        print(f"  {i}. {f.name:50s} ({size_mb:8.2f} MB)")
    
    if not files:
        print("  ⚠️  No CSV files found!")
else:
    print(f"  ⚠️  Directory not found: {data_dir}")

## 3️⃣ Data Preprocessing & Formatting

**Configure preprocessing parameters and run the pipeline.**

You can adjust these settings and re-run this section at any time.

In [ ]:
# ============================================================================
# 🎛️ PREPROCESSING CONFIGURATION - ADJUST THESE PARAMETERS
# ============================================================================

# Imputation settings
IMPUTE_ENABLED = True           # Enable/disable missing value imputation
IMPUTE_METHOD = 'median'        # Options: 'mean', 'median', 'zero'
IMPUTE_THRESHOLD = 0.5          # Drop columns with >50% missing values

# Feature engineering
SMOOTH_STATIC = False           # Apply smoothing to static features
CAGR_YEARS = 3                  # Number of years for CAGR calculation

# Display configuration
print("🎛️ Preprocessing Configuration:")
print("=" * 80)
print(f"  Imputation Enabled:     {IMPUTE_ENABLED}")
print(f"  Imputation Method:      {IMPUTE_METHOD}")
print(f"  Imputation Threshold:   {IMPUTE_THRESHOLD} ({IMPUTE_THRESHOLD*100:.0f}% missing)")
print(f"  Smooth Static Features: {SMOOTH_STATIC}")
print(f"  CAGR Years:             {CAGR_YEARS}")
print("=" * 80)

In [ ]:
# Run preprocessing pipeline
print("\n🔄 Running preprocessing pipeline...\n")

df_features = data_cleaner.run_preprocessing(
    input_dir=input_dir,
    market=market,
    impute=IMPUTE_ENABLED,
    impute_method=IMPUTE_METHOD,
    impute_threshold=IMPUTE_THRESHOLD,
    smooth_static=SMOOTH_STATIC,
    cagr_years=CAGR_YEARS
)

print("\n✅ Preprocessing complete!")
display_dataframe_summary(df_features, "Processed Features")

In [ ]:
# Preview processed data
print("\n📊 Data Preview:")
display(df_features.head(10))

print(f"\n📈 Data Shape: {df_features.shape[0]:,} rows × {df_features.shape[1]:,} columns")

In [ ]:
# Prepare time-series datasets
print("\n🔄 Preparing time-series datasets...\n")

df_all, df_latest = data_cleaner.prepare_time_data(df_features, market)

print("✅ Time-series preparation complete!\n")
display_dataframe_summary(df_all, "All Time Periods")
display_dataframe_summary(df_latest, "Latest Snapshot")

# Save to state
state.save('df_features', df_features)
state.save('df_all', df_all)
state.save('df_latest', df_latest)

## 4️⃣ Outlier Detection & Handling

**Visualize outliers and understand their impact.**

In [ ]:
# Identify feature types
numeric_cols = df_latest.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['company_id', 'gvkey', 'year', 'fyear', 'cluster', 'label', 'latest_year']
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

# Categorize features
static_features = [f for f in feature_cols if not any(x in f for x in ['_trend', '_volatility', '_cagr', '_growth'])]
dynamic_features = [f for f in feature_cols if any(x in f for x in ['_trend', '_volatility', '_cagr', '_growth'])]

print(f"\n📊 Feature Overview:")
print("=" * 80)
print(f"  Total Features:        {len(feature_cols)}")
print(f"  Static Features:       {len(static_features)}")
print(f"  Dynamic Features:      {len(dynamic_features)}")
print("=" * 80)

print(f"\n📌 Static Features ({len(static_features)}):")
for i, f in enumerate(static_features[:15], 1):
    print(f"  {i:2d}. {f}")
if len(static_features) > 15:
    print(f"      ... and {len(static_features) - 15} more")

print(f"\n📈 Dynamic Features ({len(dynamic_features)}):")
for i, f in enumerate(dynamic_features[:15], 1):
    print(f"  {i:2d}. {f}")
if len(dynamic_features) > 15:
    print(f"      ... and {len(dynamic_features) - 15} more")

# Save feature lists
state.save('feature_cols', feature_cols)
state.save('static_features', static_features)
state.save('dynamic_features', dynamic_features)

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(df, features, multiplier=1.5):
    """Detect outliers using IQR method"""
    outlier_info = []
    
    for col in features:
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        outlier_pct = outliers / len(df) * 100
        
        if outliers > 0:
            outlier_info.append({
                'Feature': col,
                'Outliers': outliers,
                'Percentage': outlier_pct,
                'Lower_Bound': lower_bound,
                'Upper_Bound': upper_bound
            })
    
    return pd.DataFrame(outlier_info).sort_values('Percentage', ascending=False)

# Detect outliers
outlier_summary = detect_outliers_iqr(df_latest, feature_cols[:20])  # First 20 features

print("\n🔍 Outlier Detection (Top 15 features with most outliers):")
print("=" * 80)
if len(outlier_summary) > 0:
    display(outlier_summary.head(15))
else:
    print("  ✅ No outliers detected!")
print("=" * 80)

In [ ]:
# Visualize outliers for top features
if len(outlier_summary) > 0:
    top_outlier_features = outlier_summary.head(6)['Feature'].tolist()
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, feature in enumerate(top_outlier_features):
        ax = axes[idx]
        
        # Box plot
        df_latest[feature].plot(kind='box', ax=ax, vert=False, color='steelblue')
        ax.set_xlabel('Value', fontsize=10)
        ax.set_title(f'{feature}\n({outlier_summary[outlier_summary["Feature"]==feature]["Percentage"].values[0]:.1f}% outliers)',
                    fontsize=11, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.suptitle('Outlier Visualization (Box Plots)', y=1.02, fontsize=14, fontweight='bold')
    plt.show()
else:
    print("✅ No outliers to visualize")

## 5️⃣ Feature Selection

**Choose which features to use for clustering.**

Options:
- `'static'`: Current financial ratios only
- `'dynamic'`: Time-series features only (trends, volatility, growth)
- `'combined'`: All features together

In [ ]:
# ============================================================================
# 🎛️ FEATURE SELECTION - ADJUST THIS PARAMETER
# ============================================================================

FEATURE_SET = 'combined'  # Options: 'static', 'dynamic', 'combined'

# Select features based on choice
if FEATURE_SET == 'static':
    selected_features = static_features
elif FEATURE_SET == 'dynamic':
    selected_features = dynamic_features
else:  # combined
    selected_features = feature_cols

print(f"\n🎯 Feature Selection:")
print("=" * 80)
print(f"  Selected Set:    {FEATURE_SET.upper()}")
print(f"  Features Count:  {len(selected_features)}")
print("=" * 80)

if len(selected_features) <= 20:
    print(f"\n📋 Selected Features:")
    for i, f in enumerate(selected_features, 1):
        print(f"  {i:2d}. {f}")
else:
    print(f"\n📋 Selected Features (first 20 of {len(selected_features)}):")
    for i, f in enumerate(selected_features[:20], 1):
        print(f"  {i:2d}. {f}")
    print(f"      ... and {len(selected_features) - 20} more")

# Save selection
state.save('selected_features', selected_features)
state.save('feature_set_type', FEATURE_SET)

In [ ]:
# Feature correlation analysis
top_n = min(15, len(selected_features))
corr_features = selected_features[:top_n]

df_corr = df_latest[corr_features].fillna(df_latest[corr_features].median())
corr_matrix = df_corr.corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title(f'Feature Correlation Matrix\n({FEATURE_SET.capitalize()} - Top {top_n} Features)',
         fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find highly correlated pairs
high_corr_threshold = 0.8
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > high_corr_threshold:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print(f"\n⚠️  Highly Correlated Pairs (|r| > {high_corr_threshold}):")
    for feat1, feat2, corr_val in high_corr_pairs:
        print(f"  • {feat1:30s} <-> {feat2:30s}  (r = {corr_val:6.3f})")
else:
    print(f"\n✅ No highly correlated pairs found (threshold: {high_corr_threshold})")

## 6️⃣ PCA Configuration (Toggle)

**Enable or disable PCA and configure parameters.**

PCA (Principal Component Analysis) reduces dimensionality while preserving variance.

**When to use PCA:**
- Many features (>10)
- High feature correlation
- Curse of dimensionality issues

**When to skip PCA:**
- Few features (<10)
- Need interpretability
- Features already well-separated

In [ ]:
# ============================================================================
# 🎛️ PCA CONFIGURATION - ADJUST THESE PARAMETERS
# ============================================================================

USE_PCA = True                  # Toggle PCA on/off
PCA_VARIANCE_THRESHOLD = 0.85   # Keep components explaining 85% variance
# OR specify exact number of components:
# PCA_N_COMPONENTS = 10         # Uncomment to use fixed number

print(f"\n🎛️ PCA Configuration:")
print("=" * 80)
print(f"  PCA Enabled:           {USE_PCA}")
if USE_PCA:
    print(f"  Variance Threshold:    {PCA_VARIANCE_THRESHOLD} ({PCA_VARIANCE_THRESHOLD*100:.0f}%)")
print("=" * 80)

In [ ]:
# Prepare clustering dataset
df_cluster = df_latest[selected_features].copy()

# Handle missing values
df_cluster = df_cluster.fillna(df_cluster.median())

print(f"\n📊 Clustering Dataset:")
print(f"  Samples:         {len(df_cluster):,}")
print(f"  Features:        {len(selected_features)}")
print(f"  Missing Values:  {df_cluster.isnull().sum().sum()}")
print(f"  Memory Usage:    {df_cluster.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Apply PCA if enabled
if USE_PCA:
    print("\n🔄 Applying PCA...\n")
    
    # First scale the data
    scaler_for_pca = StandardScaler()
    X_scaled_for_pca = scaler_for_pca.fit_transform(df_cluster)
    
    # Apply PCA
    pca = PCA(n_components=PCA_VARIANCE_THRESHOLD, random_state=42)
    X_pca = pca.fit_transform(X_scaled_for_pca)
    
    n_components = pca.n_components_
    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)
    
    print(f"✅ PCA Applied!")
    print(f"  Original Features:     {len(selected_features)}")
    print(f"  PCA Components:        {n_components}")
    print(f"  Dimensionality Reduction: {(1 - n_components/len(selected_features))*100:.1f}%")
    print(f"  Explained Variance:    {cumulative_variance[-1]:.1%}")
    
    # Create DataFrame with PCA components
    pca_columns = [f'PC{i+1}' for i in range(n_components)]
    df_pca = pd.DataFrame(X_pca, columns=pca_columns, index=df_cluster.index)
    
    # Visualize explained variance
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Individual variance
    ax1.bar(range(1, n_components+1), explained_variance, alpha=0.7, color='steelblue')
    ax1.set_xlabel('Principal Component', fontsize=12)
    ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
    ax1.set_title('PCA: Individual Component Variance', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    # Cumulative variance
    ax2.plot(range(1, n_components+1), cumulative_variance, 'o-', linewidth=2, markersize=6, color='darkgreen')
    ax2.axhline(y=PCA_VARIANCE_THRESHOLD, color='red', linestyle='--', 
               label=f'Threshold: {PCA_VARIANCE_THRESHOLD:.0%}')
    ax2.set_xlabel('Number of Components', fontsize=12)
    ax2.set_ylabel('Cumulative Explained Variance', fontsize=12)
    ax2.set_title('PCA: Cumulative Variance', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.set_ylim([0, 1.05])
    
    plt.tight_layout()
    plt.show()
    
    # Show top feature loadings for first 3 PCs
    print(f"\n📊 Top 5 Feature Loadings per Component (first 3 PCs):")
    print("=" * 80)
    
    for pc_idx in range(min(3, n_components)):
        loadings = pd.Series(pca.components_[pc_idx], index=selected_features)
        top_loadings = loadings.abs().nlargest(5)
        
        print(f"\n  PC{pc_idx+1} (Variance: {explained_variance[pc_idx]:.1%}):")
        for feat in top_loadings.index:
            loading_val = loadings[feat]
            print(f"    {feat:40s}  {loading_val:7.3f}")
    
    print("=" * 80)
    
    # Use PCA components for clustering
    clustering_data = df_pca
    clustering_features = pca_columns
    
    # Save PCA results
    state.save('pca_model', pca)
    state.save('pca_scaler', scaler_for_pca)
    state.save('pca_components', df_pca)
    
else:
    print("\n⏭️  PCA disabled - using original features")
    clustering_data = df_cluster
    clustering_features = selected_features

print(f"\n✅ Data ready for clustering!")
print(f"  Shape: {clustering_data.shape}")

## 7️⃣ Data Scaling & Preparation

**Scale features for clustering algorithms.**

In [ ]:
# ============================================================================
# 🎛️ SCALING CONFIGURATION
# ============================================================================

SCALER_TYPE = 'standard'  # Options: 'standard' (Z-score), 'robust' (median/IQR)

print(f"\n🎛️ Scaling Configuration:")
print(f"  Scaler Type: {SCALER_TYPE.upper()}")

In [ ]:
# Apply scaling
if SCALER_TYPE == 'robust':
    scaler = RobustScaler()
else:
    scaler = StandardScaler()

X_scaled = scaler.fit_transform(clustering_data)

print(f"\n✅ Scaling applied using {SCALER_TYPE.upper()}Scaler")
print(f"  Scaled Data Shape:  {X_scaled.shape}")
print(f"  Mean:               {X_scaled.mean():.6f}")
print(f"  Std:                {X_scaled.std():.6f}")

# Save scaler
state.save('scaler', scaler)
state.save('X_scaled', X_scaled)
state.save('clustering_features', clustering_features)

## 8️⃣ K-Means Clustering

**Interactive K-Means with adjustable parameters.**

In [ ]:
# Find optimal K using multiple methods
K_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []
davies_bouldin_scores = []

print("\n🔄 Testing different K values...\n")

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    calinski_scores.append(calinski_harabasz_score(X_scaled, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled, labels))
    
    print(f"  K={k:2d}  |  Inertia: {kmeans.inertia_:8.2f}  |  "
          f"Silhouette: {silhouette_scores[-1]:6.3f}  |  "
          f"Calinski-H: {calinski_scores[-1]:8.2f}  |  "
          f"Davies-Bouldin: {davies_bouldin_scores[-1]:6.3f}")

print("\n✅ K-value testing complete")

In [ ]:
# Visualize optimal K selection
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Elbow curve
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia (WCSS)', fontsize=12)
ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Silhouette score
ax2.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
best_sil_k = list(K_range)[np.argmax(silhouette_scores)]
ax2.axvline(best_sil_k, color='red', linestyle='--', alpha=0.7, 
           label=f'Best K={best_sil_k}')
ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score (higher is better)', fontsize=12)
ax2.set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Calinski-Harabasz score
ax3.plot(K_range, calinski_scores, 'mo-', linewidth=2, markersize=8)
best_cal_k = list(K_range)[np.argmax(calinski_scores)]
ax3.axvline(best_cal_k, color='red', linestyle='--', alpha=0.7,
           label=f'Best K={best_cal_k}')
ax3.set_xlabel('Number of Clusters (K)', fontsize=12)
ax3.set_ylabel('Calinski-Harabasz Score (higher is better)', fontsize=12)
ax3.set_title('Calinski-Harabasz Index', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Davies-Bouldin score
ax4.plot(K_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
best_db_k = list(K_range)[np.argmin(davies_bouldin_scores)]
ax4.axvline(best_db_k, color='green', linestyle='--', alpha=0.7,
           label=f'Best K={best_db_k}')
ax4.set_xlabel('Number of Clusters (K)', fontsize=12)
ax4.set_ylabel('Davies-Bouldin Score (lower is better)', fontsize=12)
ax4.set_title('Davies-Bouldin Index', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

plt.tight_layout()
plt.suptitle('K-Means: Optimal K Selection', y=1.01, fontsize=16, fontweight='bold')
plt.show()

print(f"\n📊 Recommendations:")
print(f"  Best Silhouette:       K = {best_sil_k} (Score: {max(silhouette_scores):.3f})")
print(f"  Best Calinski-Harabasz: K = {best_cal_k} (Score: {max(calinski_scores):.2f})")
print(f"  Best Davies-Bouldin:    K = {best_db_k} (Score: {min(davies_bouldin_scores):.3f})")

In [ ]:
# ============================================================================
# 🎛️ K-MEANS CONFIGURATION - ADJUST THESE PARAMETERS
# ============================================================================

KMEANS_K = best_sil_k        # Number of clusters (use best from above or set manually)
KMEANS_RANDOM_STATE = 42     # Random seed for reproducibility
KMEANS_N_INIT = 10           # Number of initializations
KMEANS_MAX_ITER = 300        # Maximum iterations

print(f"\n🎛️ K-Means Configuration:")
print("=" * 80)
print(f"  K (clusters):      {KMEANS_K}")
print(f"  Random State:      {KMEANS_RANDOM_STATE}")
print(f"  N Init:            {KMEANS_N_INIT}")
print(f"  Max Iterations:    {KMEANS_MAX_ITER}")
print("=" * 80)

In [ ]:
# Fit K-Means
print(f"\n🔄 Training K-Means with K={KMEANS_K}...\n")

kmeans = KMeans(
    n_clusters=KMEANS_K,
    random_state=KMEANS_RANDOM_STATE,
    n_init=KMEANS_N_INIT,
    max_iter=KMEANS_MAX_ITER
)

kmeans_labels = kmeans.fit_predict(X_scaled)

# Calculate metrics
kmeans_inertia = kmeans.inertia_
kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)
kmeans_calinski = calinski_harabasz_score(X_scaled, kmeans_labels)
kmeans_davies_bouldin = davies_bouldin_score(X_scaled, kmeans_labels)

print("✅ K-Means clustering complete!\n")
print("📊 Quality Metrics:")
print("=" * 80)
print(f"  Inertia:              {kmeans_inertia:.2f}")
print(f"  Silhouette Score:     {kmeans_silhouette:.3f}  (range: -1 to 1, higher is better)")
print(f"  Calinski-Harabasz:    {kmeans_calinski:.2f}  (higher is better)")
print(f"  Davies-Bouldin:       {kmeans_davies_bouldin:.3f}  (lower is better)")
print(f"  Iterations:           {kmeans.n_iter_}")
print("=" * 80)

# Add labels to dataframe
df_kmeans = df_latest.copy()
df_kmeans['cluster_kmeans'] = kmeans_labels

# Save results
state.save('kmeans_model', kmeans)
state.save('kmeans_labels', kmeans_labels)
state.save('df_kmeans', df_kmeans)

In [ ]:
# Cluster distribution
cluster_counts = plot_cluster_distribution(
    df_kmeans, 
    cluster_col='cluster_kmeans',
    title=f'K-Means Cluster Distribution (K={KMEANS_K})'
)

print(f"\n📊 Cluster Sizes:")
for cluster_id in sorted(cluster_counts.keys()):
    count = cluster_counts[cluster_id]
    pct = count / len(df_kmeans) * 100
    print(f"  Cluster {cluster_id}: {count:5d} companies ({pct:5.1f}%)")

In [ ]:
# Cluster profiling
if USE_PCA:
    # Profile using PCA components
    df_profile = df_kmeans.copy()
    for col in clustering_features:
        df_profile[col] = clustering_data[col]
    
    cluster_profiles = df_profile.groupby('cluster_kmeans')[clustering_features].mean()
    
    print("\n📊 Cluster Profiles (PCA Component Means):")
    display(cluster_profiles)
    
    # Also show original feature profiles
    original_profiles = df_kmeans.groupby('cluster_kmeans')[selected_features].mean()
    
    print("\n📊 Cluster Profiles (Original Feature Means - Top 10 features):")
    display(original_profiles[selected_features[:10]])
else:
    # Profile using original features
    cluster_profiles = df_kmeans.groupby('cluster_kmeans')[selected_features].mean()
    
    print("\n📊 Cluster Profiles (Feature Means):")
    display(cluster_profiles)

# Save profiles
state.save('kmeans_profiles', cluster_profiles)

In [ ]:
# Visualize clusters in 2D PCA space
pca_viz = PCA(n_components=2, random_state=42)
X_pca_2d = pca_viz.fit_transform(X_scaled)

explained_var_viz = pca_viz.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(14, 10))

# Plot each cluster
colors = plt.cm.nipy_spectral(np.linspace(0, 1, KMEANS_K))

for cluster_id in range(KMEANS_K):
    mask = kmeans_labels == cluster_id
    ax.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
              c=[colors[cluster_id]], label=f'Cluster {cluster_id}',
              alpha=0.6, s=60, edgecolors='k', linewidth=0.5)

# Plot centroids
centroids_pca = pca_viz.transform(kmeans.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
          c='red', marker='X', s=400, edgecolors='black',
          linewidth=2, label='Centroids', zorder=10)

ax.set_xlabel(f'PC1 ({explained_var_viz[0]:.1%} variance)', fontsize=13)
ax.set_ylabel(f'PC2 ({explained_var_viz[1]:.1%} variance)', fontsize=13)
ax.set_title(f'K-Means Clusters in 2D PCA Space (K={KMEANS_K})\nTotal Variance: {explained_var_viz.sum():.1%}',
            fontsize=15, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9️⃣ Hierarchical Clustering

**Interactive hierarchical clustering with dendrograms.**

In [ ]:
# ============================================================================
# 🎛️ HIERARCHICAL CLUSTERING CONFIGURATION
# ============================================================================

HIER_LINKAGE = 'ward'        # Options: 'ward', 'complete', 'average', 'single'
HIER_N_CLUSTERS = KMEANS_K   # Use same K as K-Means for comparison
DENDROGRAM_SAMPLE = 500      # Number of samples for dendrogram (speeds up visualization)

print(f"\n🎛️ Hierarchical Clustering Configuration:")
print("=" * 80)
print(f"  Linkage Method:    {HIER_LINKAGE}")
print(f"  N Clusters:        {HIER_N_CLUSTERS}")
print(f"  Dendrogram Sample: {DENDROGRAM_SAMPLE}")
print("=" * 80)

In [ ]:
# Create dendrogram
print(f"\n🔄 Generating dendrogram (using {min(DENDROGRAM_SAMPLE, len(X_scaled))} samples)...\n")

# Sample data for dendrogram
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), min(DENDROGRAM_SAMPLE, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

# Calculate linkage
Z = linkage(X_sample, method=HIER_LINKAGE)

# Plot dendrogram
plt.figure(figsize=(16, 8))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_font_size=11,
          show_leaf_counts=True, no_labels=True)
plt.title(f'Hierarchical Clustering Dendrogram ({HIER_LINKAGE.capitalize()} Linkage)',
         fontsize=15, fontweight='bold')
plt.xlabel('Cluster Size', fontsize=13)
plt.ylabel('Distance', fontsize=13)

# Add cut line
if HIER_N_CLUSTERS <= len(Z):
    cut_height = Z[-HIER_N_CLUSTERS, 2]
    plt.axhline(y=cut_height, c='red', linestyle='--', linewidth=2,
               label=f'Cut for {HIER_N_CLUSTERS} clusters')
    plt.legend(fontsize=12)

plt.tight_layout()
plt.show()

print("✅ Dendrogram generated")

In [ ]:
# Fit hierarchical clustering on full dataset
print(f"\n🔄 Fitting hierarchical clustering on full dataset...\n")

hierarchical = AgglomerativeClustering(
    n_clusters=HIER_N_CLUSTERS,
    linkage=HIER_LINKAGE
)

hier_labels = hierarchical.fit_predict(X_scaled)

# Calculate metrics
hier_silhouette = silhouette_score(X_scaled, hier_labels)
hier_calinski = calinski_harabasz_score(X_scaled, hier_labels)
hier_davies_bouldin = davies_bouldin_score(X_scaled, hier_labels)

print("✅ Hierarchical clustering complete!\n")
print("📊 Quality Metrics:")
print("=" * 80)
print(f"  Silhouette Score:     {hier_silhouette:.3f}")
print(f"  Calinski-Harabasz:    {hier_calinski:.2f}")
print(f"  Davies-Bouldin:       {hier_davies_bouldin:.3f}")
print("=" * 80)

# Add to dataframe
df_hier = df_latest.copy()
df_hier['cluster_hier'] = hier_labels

# Save results
state.save('hier_labels', hier_labels)
state.save('df_hier', df_hier)

In [ ]:
# Cluster distribution
hier_counts = plot_cluster_distribution(
    df_hier,
    cluster_col='cluster_hier',
    title=f'Hierarchical Cluster Distribution ({HIER_LINKAGE})'
)

print(f"\n📊 Cluster Sizes:")
for cluster_id in sorted(hier_counts.keys()):
    count = hier_counts[cluster_id]
    pct = count / len(df_hier) * 100
    print(f"  Cluster {cluster_id}: {count:5d} companies ({pct:5.1f}%)")

## 🔟 DBSCAN Clustering

**Density-based clustering with automatic outlier detection.**

In [ ]:
# K-distance plot to determine eps
MIN_SAMPLES_DEFAULT = 5

print(f"\n🔄 Computing k-distance plot (k={MIN_SAMPLES_DEFAULT})...\n")

neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES_DEFAULT)
neighbors.fit(X_scaled)
distances, indices = neighbors.kneighbors(X_scaled)

# Sort distances
k_distances = np.sort(distances[:, -1])[::-1]

# Plot k-distance
plt.figure(figsize=(14, 6))
plt.plot(k_distances, linewidth=1.5, color='steelblue')
plt.xlabel('Data Points (sorted by distance)', fontsize=12)
plt.ylabel(f'{MIN_SAMPLES_DEFAULT}-th Nearest Neighbor Distance', fontsize=12)
plt.title('K-Distance Plot\n(Look for "elbow" to determine optimal eps)',
         fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Add suggested eps line
suggested_eps = np.percentile(k_distances, 95)
plt.axhline(y=suggested_eps, color='red', linestyle='--', linewidth=2,
           label=f'Suggested eps (95th %ile): {suggested_eps:.3f}')
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"💡 Suggested eps value: {suggested_eps:.3f}")

In [ ]:
# ============================================================================
# 🎛️ DBSCAN CONFIGURATION - ADJUST THESE PARAMETERS
# ============================================================================

DBSCAN_EPS = suggested_eps   # Epsilon (neighborhood radius)
DBSCAN_MIN_SAMPLES = 5       # Minimum samples per cluster

print(f"\n🎛️ DBSCAN Configuration:")
print("=" * 80)
print(f"  Eps (ε):          {DBSCAN_EPS:.3f}")
print(f"  Min Samples:      {DBSCAN_MIN_SAMPLES}")
print("=" * 80)

In [ ]:
# Fit DBSCAN
print(f"\n🔄 Running DBSCAN...\n")

dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Count clusters and outliers
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_outliers = list(dbscan_labels).count(-1)
n_outliers_pct = n_outliers / len(dbscan_labels) * 100

print("✅ DBSCAN complete!\n")
print("📊 Results:")
print("=" * 80)
print(f"  Clusters Found:       {n_clusters_dbscan}")
print(f"  Outliers:             {n_outliers} ({n_outliers_pct:.1f}%)")
print(f"  Clustered Points:     {len(dbscan_labels) - n_outliers} ({100-n_outliers_pct:.1f}%)")
print("=" * 80)

# Calculate metrics (excluding outliers)
if n_clusters_dbscan > 1 and n_outliers < len(dbscan_labels):
    mask_clustered = dbscan_labels != -1
    dbscan_silhouette = silhouette_score(X_scaled[mask_clustered], dbscan_labels[mask_clustered])
    dbscan_calinski = calinski_harabasz_score(X_scaled[mask_clustered], dbscan_labels[mask_clustered])
    dbscan_davies_bouldin = davies_bouldin_score(X_scaled[mask_clustered], dbscan_labels[mask_clustered])
    
    print(f"\n📊 Quality Metrics (excluding outliers):")
    print("=" * 80)
    print(f"  Silhouette Score:     {dbscan_silhouette:.3f}")
    print(f"  Calinski-Harabasz:    {dbscan_calinski:.2f}")
    print(f"  Davies-Bouldin:       {dbscan_davies_bouldin:.3f}")
    print("=" * 80)
elif n_clusters_dbscan <= 1:
    print(f"\n⚠️  Only {n_clusters_dbscan} cluster(s) found. Try adjusting eps or min_samples.")
else:
    print(f"\n⚠️  All points are outliers. Try adjusting eps or min_samples.")

# Add to dataframe
df_dbscan = df_latest.copy()
df_dbscan['cluster_dbscan'] = dbscan_labels

# Save results
state.save('dbscan_labels', dbscan_labels)
state.save('df_dbscan', df_dbscan)

In [ ]:
# Cluster distribution (excluding outliers)
if n_clusters_dbscan > 0:
    df_dbscan_clustered = df_dbscan[df_dbscan['cluster_dbscan'] != -1]
    
    if len(df_dbscan_clustered) > 0:
        dbscan_counts = plot_cluster_distribution(
            df_dbscan_clustered,
            cluster_col='cluster_dbscan',
            title=f'DBSCAN Cluster Distribution (eps={DBSCAN_EPS:.3f}, min_samples={DBSCAN_MIN_SAMPLES})'
        )
        
        print(f"\n📊 Cluster Sizes (excluding {n_outliers} outliers):")
        for cluster_id in sorted(dbscan_counts.keys()):
            count = dbscan_counts[cluster_id]
            pct = count / len(df_dbscan) * 100
            print(f"  Cluster {cluster_id}: {count:5d} companies ({pct:5.1f}%)")
else:
    print("\n⚠️  No clusters to visualize")

## 1️⃣1️⃣ Algorithm Comparison

**Compare all clustering algorithms side-by-side.**

In [ ]:
# Compile metrics from all algorithms
comparison_data = []

# K-Means
comparison_data.append({
    'Algorithm': 'K-Means',
    'N_Clusters': KMEANS_K,
    'Silhouette': kmeans_silhouette,
    'Calinski-Harabasz': kmeans_calinski,
    'Davies-Bouldin': kmeans_davies_bouldin
})

# Hierarchical
comparison_data.append({
    'Algorithm': 'Hierarchical',
    'N_Clusters': HIER_N_CLUSTERS,
    'Silhouette': hier_silhouette,
    'Calinski-Harabasz': hier_calinski,
    'Davies-Bouldin': hier_davies_bouldin
})

# DBSCAN (if valid)
if n_clusters_dbscan > 1 and n_outliers < len(dbscan_labels):
    comparison_data.append({
        'Algorithm': 'DBSCAN',
        'N_Clusters': n_clusters_dbscan,
        'Silhouette': dbscan_silhouette,
        'Calinski-Harabasz': dbscan_calinski,
        'Davies-Bouldin': dbscan_davies_bouldin
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n📊 Algorithm Comparison:")
print("=" * 80)
display(df_comparison)
print("\nInterpretation:")
print("  • Silhouette:       Higher is better (range: -1 to 1)")
print("  • Calinski-Harabasz: Higher is better")
print("  • Davies-Bouldin:    Lower is better")
print("=" * 80)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Silhouette
df_comparison.plot(x='Algorithm', y='Silhouette', kind='bar', ax=axes[0],
                  color='steelblue', alpha=0.7, legend=False)
axes[0].set_ylabel('Silhouette Score', fontsize=12)
axes[0].set_title('Silhouette Score\n(higher is better)', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Calinski-Harabasz
df_comparison.plot(x='Algorithm', y='Calinski-Harabasz', kind='bar', ax=axes[1],
                  color='orange', alpha=0.7, legend=False)
axes[1].set_ylabel('Calinski-Harabasz Score', fontsize=12)
axes[1].set_title('Calinski-Harabasz Score\n(higher is better)', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

# Davies-Bouldin
df_comparison.plot(x='Algorithm', y='Davies-Bouldin', kind='bar', ax=axes[2],
                  color='crimson', alpha=0.7, legend=False)
axes[2].set_ylabel('Davies-Bouldin Score', fontsize=12)
axes[2].set_title('Davies-Bouldin Score\n(lower is better)', fontsize=13, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.suptitle('Algorithm Comparison', y=1.02, fontsize=15, fontweight='bold')
plt.show()

In [ ]:
# Calculate Adjusted Rand Index (agreement between algorithms)
algorithms = ['K-Means', 'Hierarchical']
labels_list = [kmeans_labels, hier_labels]

if n_clusters_dbscan > 1:
    algorithms.append('DBSCAN')
    labels_list.append(dbscan_labels)

n_algs = len(algorithms)
ari_matrix = np.ones((n_algs, n_algs))

for i in range(n_algs):
    for j in range(i+1, n_algs):
        ari = adjusted_rand_score(labels_list[i], labels_list[j])
        ari_matrix[i, j] = ari
        ari_matrix[j, i] = ari

df_ari = pd.DataFrame(ari_matrix, index=algorithms, columns=algorithms)

print("\n📊 Adjusted Rand Index (Algorithm Agreement):")
print("=" * 80)
display(df_ari)
print("\nInterpretation:")
print("  • 1.0 = Perfect agreement")
print("  • 0.0 = Random agreement")
print("  • Negative = Less agreement than random")
print("=" * 80)

# Visualize ARI
plt.figure(figsize=(10, 8))
sns.heatmap(df_ari, annot=True, fmt='.3f', cmap='RdYlGn',
           vmin=0, vmax=1, square=True, linewidths=1,
           cbar_kws={'label': 'ARI Score'})
plt.title('Algorithm Agreement (Adjusted Rand Index)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visual comparison in PCA space
fig, axes = plt.subplots(1, len(algorithms), figsize=(7*len(algorithms), 6))

if len(algorithms) == 1:
    axes = [axes]

for idx, (alg, labels) in enumerate(zip(algorithms, labels_list)):
    ax = axes[idx]
    
    # Handle DBSCAN outliers
    unique_labels = set(labels)
    n_clust = len(unique_labels) - (1 if -1 in unique_labels else 0)
    
    colors = plt.cm.nipy_spectral(np.linspace(0, 1, max(n_clust, 2)))
    
    for k in range(n_clust):
        class_member_mask = (labels == k)
        ax.scatter(X_pca_2d[class_member_mask, 0], X_pca_2d[class_member_mask, 1],
                  c=[colors[k]], alpha=0.6, s=40, edgecolors='k', linewidth=0.3,
                  label=f'Cluster {k}')
    
    # Plot outliers for DBSCAN
    if -1 in labels:
        outlier_mask = (labels == -1)
        ax.scatter(X_pca_2d[outlier_mask, 0], X_pca_2d[outlier_mask, 1],
                  c='gray', alpha=0.3, s=25, marker='x',
                  label='Outliers')
    
    ax.set_xlabel(f'PC1 ({explained_var_viz[0]:.1%})', fontsize=11)
    ax.set_ylabel(f'PC2 ({explained_var_viz[1]:.1%})', fontsize=11)
    ax.set_title(f'{alg}\n({n_clust} clusters)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=9, ncol=2)

plt.tight_layout()
plt.suptitle('Algorithm Comparison in 2D PCA Space', y=1.01, fontsize=15, fontweight='bold')
plt.show()

## 1️⃣2️⃣ Results Export

**Save cluster assignments and visualizations.**

In [ ]:
# Create output directory
output_dir = PROJECT_ROOT / 'output' / market / 'notebooks' / 'master_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Output Directory: {output_dir}\n")

# Export cluster assignments
df_export = df_latest.copy()
df_export['cluster_kmeans'] = kmeans_labels
df_export['cluster_hierarchical'] = hier_labels
df_export['cluster_dbscan'] = dbscan_labels

export_file = output_dir / 'cluster_assignments.csv'
df_export.to_csv(export_file, index=False)
print(f"✅ Exported cluster assignments: {export_file.name}")

# Export comparison metrics
metrics_file = output_dir / 'algorithm_comparison.csv'
df_comparison.to_csv(metrics_file, index=False)
print(f"✅ Exported comparison metrics: {metrics_file.name}")

# Export ARI matrix
ari_file = output_dir / 'algorithm_agreement_ari.csv'
df_ari.to_csv(ari_file)
print(f"✅ Exported ARI matrix: {ari_file.name}")

# Export configuration summary
config_summary = {
    'Feature Set': FEATURE_SET,
    'Number of Features': len(selected_features),
    'PCA Enabled': USE_PCA,
    'PCA Components': n_components if USE_PCA else 'N/A',
    'Scaler Type': SCALER_TYPE,
    'K-Means K': KMEANS_K,
    'Hierarchical Linkage': HIER_LINKAGE,
    'Hierarchical K': HIER_N_CLUSTERS,
    'DBSCAN eps': DBSCAN_EPS,
    'DBSCAN min_samples': DBSCAN_MIN_SAMPLES,
    'DBSCAN clusters': n_clusters_dbscan,
    'DBSCAN outliers': n_outliers
}

config_file = output_dir / 'configuration_summary.txt'
with open(config_file, 'w') as f:
    f.write("MASTER WORKFLOW CONFIGURATION SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    for key, value in config_summary.items():
        f.write(f"{key:30s}: {value}\n")

print(f"✅ Exported configuration: {config_file.name}")

print(f"\n" + "=" * 80)
print(f"  ✅ ALL RESULTS EXPORTED TO: {output_dir}")
print("=" * 80)

## 🎉 Workflow Complete!

### What you accomplished:

1. ✅ Loaded and preprocessed data
2. ✅ Analyzed and handled outliers
3. ✅ Selected features (static/dynamic/combined)
4. ✅ Applied PCA for dimensionality reduction (if enabled)
5. ✅ Ran K-Means clustering with optimal k-selection
6. ✅ Ran Hierarchical clustering with dendrograms
7. ✅ Ran DBSCAN with automatic outlier detection
8. ✅ Compared all algorithms side-by-side
9. ✅ Exported results and visualizations

### 🔄 Want to try something different?

**Jump back to any section and adjust parameters:**
- Change feature set (Section 5)
- Toggle PCA on/off (Section 6)
- Try different k values (Section 8)
- Adjust DBSCAN parameters (Section 10)
- Compare different configurations

### 📊 Next steps:

- Analyze cluster profiles in detail
- Investigate outliers from DBSCAN
- Run the main.py pipeline for production results
- Export findings for your Masterarbeit

---

**Happy clustering! 🎯**